In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
df = pd.read_csv(f"{path}/Q1_data.csv") # using pandas library to read the csv path

In [ ]:
# Task 2: Write your code here:
df.head() # inspecting the first 5 rows with .head()

In [ ]:
# Task 3: Write your code here:
df.info() #using .info() to check columns, non nukk count, and dtypes

In [ ]:
# Task 4: Write your code here:
df.describe() # statistical description using describe -- from here we can tell what needs scaling too

In [ ]:
# Task 5: Write your code here:
#using a function that takes in our dataframe and target column which is 'Delivery_Time'
import matplotlib.pyplot as plt # importing matplotlib to be able to visualize
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black') #specifying the number of bins in the hsitogram and the edge color to black

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time") #passing the data frame and the target column name into the function

In [ ]:
# Task 1: Write your code here:
#order id is an irrelevent column for our task therefore will be dropped (we already have indexing from 0 to n)
df = df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
# first ill check for missing values
df.isnull().sum() # this will give me the sum of nulls for each column. if i added an extra .sum() it would give me the total null values
# THIS HAD NULLS IN IT but i reran the notebook!! they trend was the null values came from the categorical columns. and
# also the target column had the most nulls

In [ ]:
# weather is categorical so i'll use fillna(df['Weather'].mode()[0])
df['Weather'].fillna(df['Weather'].mode()[0])
df.isnull().sum()

In [ ]:
# i realized that the columns with missing values are mostly categorical. maybe if i encode them first it'll be better.
# now i will fill our target
df['Delivery_Time'].fillna(df['Delivery_Time'].mean()) # i wanted to use mode but it wouldnt work
df.isnull().sum()

In [ ]:
df

In [ ]:
# double checking
df.isnull().sum().sum() # checking for any null values anywhere not in a specififc column

In [ ]:
df['Delivery_Time'].fillna(df['Delivery_Time'].mean())

In [ ]:
# im having a lot of issues with dropping the rows.not sure why.

In [ ]:
# Task 3: Write your code here:
# checking for duplicates
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
# checking if we have categorical columns (we do but just good practice)
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Encode Categorical Features
# label encoder is used for ordinal data and the data thats present has some ordinal features
from sklearn.preprocessing import LabelEncoder #import LabelEncoder

#print('data before encoding:\n', df) #show before encoding

##label_encoder = LabelEncoder() # Instantiate LabelEncoder
#df = label_encoder.fit_transform(df) # Apply fit_transform to the column

#print('\nData after encoding:\n', df) #show after encoding
# i tried one hot but it completely messed up my code


#from sklearn.preprocessing import OneHotEncoder #import OneHotEncoder

#print('data before encoding:\n', df) #show before encoding

#onehot_encoder = OneHotEncoder(sparse_output=False) # Instantiate OneHotEncoder
#df1 = onehot_encoder.fit_transform(df) # Apply fit_transform to the copied

#print('\nData after encoding:\n', df1) #show after encoding

le = LabelEncoder()
df['Weather'] = le.fit_transform(df['Weather'])
df['Traffic_Level'] = le.fit_transform(df['Traffic_Level'])
df['Time_of_Day'] = le.fit_transform(df['Time_of_Day'])
df['Vehicle_Type'] = le.fit_transform(df['Vehicle_Type'])

In [ ]:
## Task 5: Write your code here:
# applying feature scaling using standard scaler
from sklearn.preprocessing import StandardScaler #import StandardScaler

print('data before scaling:\n', df) #show before scaling
standard_scaler = StandardScaler() # Instantiate StandardScaler
df_s = standard_scaler.fit_transform(df) # Apply fit_transform

print('\nData after scaling:\n', df_s) #show after scaling

# I did the scaling but couldnt figure out why my split woulnt work. the output is below, but the rest of the code is not scaled.



In [ ]:
# Task 6: Write your code here:
# checking for target imbalance
# we already check the distribution and it is slightly skewed. in these cases we should use log transform
# im going to use stratifed kflod to address any possible problem that could occur

In [ ]:
df

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float) # drop the target column
y = df['Delivery_Time'].astype(float) # only use the target column

In [ ]:
# Task 2,3,4,5: Write your code here:
# using stratified kfold for imblances so it is properly trained and tested with the same ratios.
from sklearn.model_selection import StratifiedKFold

# Use previously generated random classification data
X, y = X.copy(), y.copy()

# Show full dataset class distribution
full_ratio = (y.value_counts(normalize=True) * 100).sort_index()
print("Full Dataset Class Distribution")
print("  y class percentages:", {k: f"{v:.2f}%" for k, v in full_ratio.items()})
print("-" * 40)

# Define Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Stratified K-Fold Cross Validation\n" + "-"*40)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)

    # showing class distribution
    train_ratio = (y_train.value_counts(normalize=True) * 100).sort_index()
    test_ratio = (y_test.value_counts(normalize=True) * 100).sort_index()

    print("  y_train class percentages:", {k: f"{v:.2f}%" for k, v in train_ratio.items()})
    print("  y_test class percentages :", {k: f"{v:.2f}%" for k, v in test_ratio.items()})

    print("-" * 40)

In [ ]:
# train a radom forest model
from sklearn.ensemble import RandomForestClassifier

In [ ]:
sklearn_models = {

  "Random Forest": RandomForestClassifier(
      n_estimators=320,  # Number of trees
      max_depth=4
  )
}

In [ ]:
all_results = {}

for name in sklearn_models:
  all_results[name] = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}

In [ ]:
n_splits = 5 # K

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  for model_name, model in sklearn_models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train) # train
    y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    all_results[model_name]['accuracy'].append(accuracy)
    all_results[model_name]['precision'].append(precision)
    all_results[model_name]['recall'].append(recall)
    all_results[model_name]['f1'].append(f1)

In [ ]:
#MAE
from sklearn.metrics import mean_absolute_error, mean_squared_error
mae = mean_absolute_error(y_test, y_pred)

print(f"MAE:  ${mae:,.2f}")

In [ ]:
# Task 1: Write your code here:
# the coefficients of ridge and lasso would tell us the feature importance. so the larger the value the more important the feature is.

coeffs = {}

coeffs['Lasso'] = models['LASSO Regression'].coef_ # weights for each feature
coeffs['Ridge'] = models['Ridge Regression'].coef_ # weights for each feature

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

# Ridge will smoothly shrink coefficients but rarely actually sets them to 0 this is called weight decay
# Lasso would push some to zero, so if i could run the cell properly the ridge would probably have more feature importance given

In [ ]:
# Task 2: Write your code here:


In [ ]:
# Task Bonus: Write your code here: